# שיעור 11 - פרוטוקול סוכן-לסוכן (A2A)


## הגדרה


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## מהו פרוטוקול A2A?

ה**פרוטוקול Agent-to-Agent (A2A)** הוא תקן פתוח שמאפשר לסוכני בינה מלאכותית לתקשר,
לגלות זה את זה, ולשתף פעולה — אפילו כשהם בנויים על מסגרות שונות או מתארחים
על ידי שירותים שונים.

מושגי מפתח:

- **גילוי** – סוכנים מפרסמים *Agent Card* שמתאר את היכולות שלהם, מה שמקל על
  סוכנים אחרים (או על האורקסטרים) למצוא את המומחה המתאים למשימה.
- **העברת הודעות** – סוכנים מחליפים הודעות מובנות דרך פרוטוקול משותף, כך ש
  בקשה מסוכן אחד יכולה להיות מובנת וממולאת על ידי אחר בלי תלות במימוש הפנימי
  שלו.
- **מחזור חיי המשימה** – A2A מגדיר מצבים כגון *submitted*, *working*, *completed*, ו
  *failed*, ומעניק לאורקסטר שקיפות מלאה לגבי אופן התקדמות משימה שהופקדה.

בשיעור זה אנו מדמים שיתוף פעולה בסגנון A2A על ידי קישור של שלושה סוכני נסיעות מתמחים
לזרימת עבודה שבה כל סוכן תורם את המומחיות שלו ומעביר את התוצאות לסוכן הבא.


## יצירת סוכני נסיעות מתמחים


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## שיתוף פעולה רב-סוכני באמצעות תזרים עבודה

אנו מחברים את שלושת הסוכנים לתזרים עבודה סדרתי שמשקף העברת הודעות A2A:

1. **CurrencyExchangeAgent** מקבל את בקשת המשתמש ומספק הנחיות להמרת מטבע.
2. **ActivityPlannerAgent** מקבל את ההקשר המועשר ומוסיף המלצות לפעילויות.
3. **TravelManagerAgent** משלב את שני הקלטים לתוך תדריך נסיעות סופי.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## הבנת A2A בסביבת ייצור

בסביבת ייצור פרוטוקול ה-A2A פותח תרחישים חזקים חוצי-שירותים:

| יכולת | תיאור |
|---|---|
| **אינטראופרביליות בין-מסגרות** | סוכן שנבנה עם מסגרת אחת יכול להעביר משימות לסוכן שנבנה בכל מסגרת אחרת התואמת ל-A2A, מה שמאפשר אינטראופרביליות אמיתית בין ארגונים. |
| **גבולות שירות** | סוכנים יכולים להתארח במיקרו-שירותים נפרדים, באזורים בענן שונים, או אפילו בארגונים שונים, ועדיין לשתף פעולה בצורה חלקה. |
| **גילוי דינמי** | אורקסטרטור יכול לשאול מאגר כרטיסי סוכן בזמן ריצה כדי למצוא את המומחה המתאים ביותר לתת-משימה נתונה. |
| **שידור והתראות דחיפה** | A2A תומך ב-Server-Sent Events (SSE) לעדכוני התקדמות בזמן אמת ובהתראות דחיפה עבור משימות ארוכות-ריצה. |

תהליך העבודה שבנינו למעלה הוא גרסה מפושטת, שמתרחשת בתוך אותו תהליך, של תבנית זו. בסביבת
פריסה אמיתית כל סוכן היה חושף נקודת קצה HTTP, מפרסם כרטיס סוכן, ומתקשר
באמצעות פרוטוקול A2A JSON-RPC.


## סיכום

בשיעור זה למדת:

1. **מהו פרוטוקול A2A** — תקן פתוח לגילוי בין-סוכנים, העברת הודעות,
   וניהול משימות.
2. **כיצד ליצור סוכנים מתמחים** — סוכן להחלפת מטבעות, סוכן מתכנן פעילויות,
   ומארגן ניהול נסיעות.
3. **כיצד לחבר סוכנים לזרימת עבודה** — שימוש ב־`WorkflowBuilder` לדמות
    העברת הודעות עוקבת בין סוכנים.
4. **כיצד A2A פועל בסביבת ייצור** — מאפשר שיתוף פעולה בין-מסגרתי ובין-שירותי
   עם גילוי דינמי ועדכונים זורמים.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
הצהרת אחריות:
מסמך זה תורגם באמצעות שירות תרגום מבוסס בינה מלאכותית Co-op Translator (https://github.com/Azure/co-op-translator). אף שאנו שואפים לדיוק, אנא שימו לב שתרגומים אוטומטיים עשויים להכיל טעויות או אי-דיוקים. יש להתייחס למסמך המקורי בשפת המקור כמקור המוסמך. עבור מידע קריטי מומלץ להשתמש בתרגום מקצועי שבוצע על־ידי מתרגם אנושי. איננו אחראים לכל אי־הבנה או פרשנות שגויה הנובעת משימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
